# Nobel Laureates: A Century of Human Excellence
### DS 220 – Project #2 | Data Analysis with Python & Pandas
---

## Motivation & Problem Space

Since 1901, the Nobel Prize has served as humanity's highest recognition of intellectual and moral achievement. Awarded annually in Physics, Chemistry, Physiology or Medicine, Literature, Peace, and Economic Sciences, the Nobel Prize represents a rare global consensus: *this person changed the world*.

But who are these people, really? Where do they come from? How old are they when their genius is recognized? Are women equally represented? Do some countries dominate? And is living a Nobel-prize-winning life actually good for your health?

### Why This Matters

- **Policy makers** designing science education programs want to know which countries invest most effectively in producing world-class researchers.
- **Sociologists** studying gender equity can measure decades of progress (or lack thereof) in a single dataset.
- **Historians of science** can trace the rise and fall of different nations' scientific influence across the 20th century.
- **Students** like us can understand what a life devoted to excellence actually looks like.

This notebook uses the **official Nobel Prize API dataset** to answer 8 compelling questions about one of the world's most prestigious institutions — and tells a data-driven story about what it means to be exceptional.

---

## Dataset Description

**Source:** [Nobel Prize API](http://api.nobelprize.org/v1/laureate.csv) — the official Nobel Foundation data feed  
**Format:** CSV, one row per laureate-prize (a person who won twice appears twice)  
**Key columns:** `firstname`, `surname`, `born`, `died`, `bornCountry`, `gender`, `year`, `category`, `motivation`  
**Coverage:** 1901 – present (~1,000+ records)

The dataset is authoritative — it comes directly from the Nobel Foundation — and is updated each October when new laureates are announced.


## Setup: Importing Libraries & Loading Data

In [ ]:
# Core data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Inline plots
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print(" Libraries imported successfully")
print(f"   NumPy  version: {np.__version__}")
print(f"   Pandas version: {pd.__version__}")


### Loading the Nobel Laureates Dataset
We load directly from the Nobel Foundation's public CSV API endpoint. If you are running this offline, place `laureates.csv` in the same directory.


In [ ]:
# Load dataset — try local file first, fall back to URL
import os

local_path = "laureates.csv"
url = "http://api.nobelprize.org/v1/laureate.csv"

if os.path.exists(local_path):
    nobel = pd.read_csv(local_path)
    print(f" Loaded from local file: {local_path}")
else:
    nobel = pd.read_csv(url)
    nobel.to_csv(local_path, index=False)
    print(f" Downloaded and cached from Nobel API")

print(f"   Shape: {nobel.shape[0]} rows × {nobel.shape[1]} columns")


---
## Exploratory Data Analysis (EDA)

Before answering our questions, we need to understand the structure of the data, check for missing values, and do necessary cleaning.


In [ ]:
# First look at the data
nobel.head(10)


In [ ]:
# Column names and data types
nobel.info()


In [ ]:
# Summary statistics for numeric columns
nobel.describe()


### Data Cleaning & Preprocessing

In [ ]:
# ── DATE PARSING ──
# Convert born/died to datetime so we can compute lifespans and ages
nobel["born_dt"]  = pd.to_datetime(nobel["born"],  errors="coerce")
nobel["died_dt"]  = pd.to_datetime(nobel["died"],  errors="coerce")

# ── DERIVED COLUMNS ──
# Birth year (numeric) for decade analysis
nobel["birth_year"] = nobel["born_dt"].dt.year

# Age at time of winning the prize
nobel["age_at_prize"] = nobel["year"] - nobel["birth_year"]

# Lifespan in years (only for those who have died)
nobel["lifespan"] = (nobel["died_dt"] - nobel["born_dt"]) / np.timedelta64(1, "Y")

# Decade the prize was awarded
nobel["decade"] = (nobel["year"] // 10) * 10

# ── MISSING VALUE REPORT ──
missing = nobel.isnull().sum()
missing_pct = (missing / len(nobel) * 100).round(1)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)

print("=== Missing Value Report ===")
print(missing_df.to_string())
print(f"\nTotal laureate records: {len(nobel)}")
print(f"Records with valid birth date: {nobel['born_dt'].notna().sum()}")
print(f"Records with valid death date: {nobel['died_dt'].notna().sum()}")
print("\n Preprocessing complete")


In [ ]:
# Check how many unique laureates vs prize entries exist
# (Some people won twice, so appear in two rows)
print(f"Total prize entries (rows):  {len(nobel)}")
print(f"Unique laureate IDs:          {nobel['id'].nunique()}")
print(f"Prize categories:             {sorted(nobel['category'].dropna().unique())}")
print(f"Years covered:                {int(nobel['year'].min())} – {int(nobel['year'].max())}")
print(f"Gender values:                {sorted(nobel['gender'].dropna().unique())}")


---
## 8 Questions About Nobel Laureates

We now answer each question with targeted pandas analysis and visualization.

---

### Question 1 — Which countries have produced the most Nobel Laureates?

Understanding national origin reveals where intellectual investment and political freedom have historically flourished.


In [ ]:
# ── Q1: Top countries by laureate count ──
# Use bornCountry as country of origin
top_countries = (
    nobel["bornCountry"]
    .dropna()
    .value_counts()
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#c9a227" if i == 0 else "#a0a0a0" if i < 3 else "#d4d4d4" 
          for i in range(len(top_countries))]
bars = ax.barh(top_countries.index[::-1], top_countries.values[::-1], color=colors[::-1], edgecolor="white")

# Value labels
for bar, val in zip(bars, top_countries.values[::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9, fontweight='bold')

ax.set_xlabel("Number of Nobel Laureates", fontsize=11)
ax.set_title("Top 15 Countries by Nobel Laureates (by Country of Birth)", fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, top_countries.values[0] * 1.12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig("q1_countries.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 5:")
for i, (country, count) in enumerate(top_countries.head(5).items(), 1):
    print(f"  {i}. {country}: {count} laureates")


**Insight:** The United States dominates Nobel Prize counts, though much of this reflects immigration — many laureates were *born* elsewhere and moved to the US. The United Kingdom and Germany follow, reflecting centuries of investment in universities and scientific institutions.

---

### Question 2 — What is the gender breakdown, and how has it changed over time?

Gender equity in Nobel Prizes is a critical indicator of societal progress in recognizing women's intellectual contributions.


In [ ]:
# ── Q2: Gender breakdown overall + by decade ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: pie chart overall
gender_counts = nobel["gender"].value_counts()
colors_pie = ["#4a90d9", "#e8734a", "#aaaaaa"]
wedges, texts, autotexts = ax1.pie(
    gender_counts.values, 
    labels=gender_counts.index,
    autopct="%1.1f%%",
    colors=colors_pie,
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight("bold")
ax1.set_title("Overall Gender Distribution
of Nobel Laureates", fontsize=13, fontweight='bold')

# Right: stacked bar by decade
gender_decade = nobel.groupby(["decade", "gender"]).size().unstack(fill_value=0)
gender_decade.plot(
    kind="bar", ax=ax2, 
    color={"male": "#4a90d9", "female": "#e8734a", "org": "#aaaaaa"},
    edgecolor="white", linewidth=0.5
)
ax2.set_xlabel("Decade", fontsize=11)
ax2.set_ylabel("Number of Laureates", fontsize=11)
ax2.set_title("Nobel Laureates by Gender per Decade", fontsize=13, fontweight='bold')
ax2.set_xticklabels([f"{int(d)}s" for d in gender_decade.index], rotation=45, ha='right')
ax2.spines[['top','right']].set_visible(False)
ax2.legend(title="Gender")

plt.tight_layout()
plt.savefig("q2_gender.png", dpi=150, bbox_inches="tight")
plt.show()

# Female laureate count by decade
female_by_decade = gender_decade.get("female", pd.Series(0, index=gender_decade.index))
print("Female laureates per decade:")
for decade, count in female_by_decade.items():
    print(f"  {int(decade)}s: {count}")


**Insight:** Women represent only ~6% of all Nobel Prizes. However, the trend is improving — the 2010s and 2020s show more female laureates than any previous decade, suggesting slow but real progress in recognizing women's scientific contributions.

---

### Question 3 — Which prize category is awarded most frequently and to whom?

Different categories have different cultures, histories, and levels of gender/geographic diversity.


In [ ]:
# ── Q3: Prizes by category ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category counts
cat_counts = nobel["category"].value_counts()
cat_colors = ["#2d6a9f","#e8734a","#4caf7d","#f0c040","#9b59b6","#e74c3c","#1abc9c"]

ax = axes[0]
bars = ax.bar(cat_counts.index, cat_counts.values, color=cat_colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel("Number of Prizes Awarded", fontsize=11)
ax.set_title("Total Prizes by Category", fontsize=13, fontweight='bold')
ax.set_xticklabels(cat_counts.index, rotation=30, ha='right')
ax.spines[['top','right']].set_visible(False)

# Female % by category
ax2 = axes[1]
female_pct = (
    nobel[nobel["gender"] == "female"]
    .groupby("category").size() /
    nobel.groupby("category").size() * 100
).sort_values(ascending=True)
colors_f = ["#e8734a" if v > 10 else "#d4a0a0" for v in female_pct.values]
ax2.barh(female_pct.index, female_pct.values, color=colors_f, edgecolor="white")
ax2.set_xlabel("% Female Laureates", fontsize=11)
ax2.set_title("Female Laureates (%) by Category", fontsize=13, fontweight='bold')
ax2.axvline(x=female_pct.mean(), color='gray', linestyle='--', alpha=0.7, label=f'Average: {female_pct.mean():.1f}%')
ax2.legend()
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig("q3_categories.png", dpi=150, bbox_inches="tight")
plt.show()

print("Category breakdown:")
for cat, count in cat_counts.items():
    pct_female = female_pct.get(cat, 0)
    print(f"  {cat:20s}: {count:4d} prizes | {pct_female:.1f}% female")


**Insight:** Medicine and Physics award the most prizes. Literature and Peace have the highest proportion of female laureates, while Physics has historically been the most male-dominated category.

---

### Question 4 — How old are laureates when they win, and has this changed over time?

Age at prize reveals whether science rewards young breakthroughs or decades of sustained contribution.


In [ ]:
# ── Q4: Age at prize ──
age_data = nobel["age_at_prize"].dropna()
age_data = age_data[(age_data > 15) & (age_data < 100)]  # sanity filter

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(age_data, bins=30, color="#2d6a9f", edgecolor="white", linewidth=0.8, alpha=0.85)
ax1.axvline(age_data.mean(), color="#e8734a", linestyle="--", linewidth=2, label=f"Mean: {age_data.mean():.1f} yrs")
ax1.axvline(age_data.median(), color="#4caf7d", linestyle="--", linewidth=2, label=f"Median: {age_data.median():.1f} yrs")
ax1.set_xlabel("Age at Time of Prize", fontsize=11)
ax1.set_ylabel("Number of Laureates", fontsize=11)
ax1.set_title("Distribution of Age at Nobel Prize", fontsize=13, fontweight='bold')
ax1.legend()
ax1.spines[['top','right']].set_visible(False)

# Mean age by decade
age_by_decade = nobel.groupby("decade")["age_at_prize"].mean().dropna()
ax2.plot(age_by_decade.index, age_by_decade.values, "o-", color="#2d6a9f", linewidth=2.5, markersize=8)
ax2.fill_between(age_by_decade.index, age_by_decade.values, alpha=0.15, color="#2d6a9f")
ax2.set_xlabel("Decade of Prize", fontsize=11)
ax2.set_ylabel("Mean Age at Prize", fontsize=11)
ax2.set_title("Average Age at Prize Award by Decade", fontsize=13, fontweight='bold')
ax2.set_xticks(age_by_decade.index)
ax2.set_xticklabels([f"{int(d)}s" for d in age_by_decade.index], rotation=45)
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig("q4_age.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Age statistics at time of prize:")
print(f"  Mean:   {age_data.mean():.1f} years")
print(f"  Median: {age_data.median():.1f} years")
print(f"  Youngest: {age_data.min():.0f} years")
print(f"  Oldest:   {age_data.max():.0f} years")

# Youngest and oldest
youngest = nobel.loc[nobel["age_at_prize"] == age_data.min(), ["firstname","surname","year","category","age_at_prize"]]
oldest   = nobel.loc[nobel["age_at_prize"] == age_data.max(), ["firstname","surname","year","category","age_at_prize"]]
print(f"\nYoungest: {youngest[['firstname','surname','age_at_prize','category']].values}")
print(f"Oldest:   {oldest[['firstname','surname','age_at_prize','category']].values}")


**Insight:** The average laureate is in their late 50s when recognized. Interestingly, the average age has *increased* over the decades — modern science requires longer careers to produce prize-worthy breakthroughs. Physics tends to reward younger winners than Medicine or Economics.

---

### Question 5 — Who are the multiple Nobel Prize winners?

Only a handful of people have achieved the extraordinary feat of winning the Nobel Prize more than once.


In [ ]:
# ── Q5: Multiple Nobel Prize winners ──
laureates = nobel.groupby(["id", "firstname", "surname"])
sizes = laureates.size()
multi_winners = sizes[sizes > 1].reset_index()
multi_winners.columns = ["id", "firstname", "surname", "prize_count"]
multi_winners = multi_winners.sort_values("prize_count", ascending=False)

print("=== Multiple Nobel Prize Winners ===")
print(multi_winners[["firstname", "surname", "prize_count"]].to_string(index=False))

# Details for each multi-winner
print("\n=== Prize Details ===")
for _, row in multi_winners.iterrows():
    prizes = nobel[nobel["id"] == row["id"]][["year", "category", "motivation"]].values
    print(f"\n{row['firstname']} {row['surname']} ({int(row['prize_count'])} prizes):")
    for prize in prizes:
        print(f"  {int(prize[0])} | {prize[1]}")


In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(9, 4))

names = multi_winners["firstname"] + " " + multi_winners["surname"]
colors = ["#c9a227","#a0a0a0","#c07820"][:len(multi_winners)]

bars = ax.barh(names, multi_winners["prize_count"].values, color=colors[::-1], edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, multi_winners["prize_count"].values):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{int(val)} prizes", va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0, multi_winners["prize_count"].max() + 0.5)
ax.set_xlabel("Number of Nobel Prizes", fontsize=11)
ax.set_title("🏅 Multiple Nobel Prize Winners — The Rarest Achievers", fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig("q5_multi.png", dpi=150, bbox_inches="tight")
plt.show()


**Insight:** Only 4 individuals and 1 organization have ever won the Nobel Prize more than once. Marie Curie remains uniquely remarkable — the only person to win in two *different scientific fields* (Physics 1903, Chemistry 1911). The International Committee of the Red Cross has won the Peace Prize three times.

---

### Question 6 — Do Nobel laureates live longer than average?

Researchers have published papers suggesting Nobel winners outlive their nominated-but-unsuccessful peers. What does our data show?


In [ ]:
# ── Q6: Lifespan analysis ──
lifespan_data = nobel["lifespan"].dropna()
lifespan_data = lifespan_data[(lifespan_data > 20) & (lifespan_data < 115)]  # sanity

# US life expectancy reference (historical average ~70)
us_life_exp = 72.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(lifespan_data, bins=30, color="#4caf7d", edgecolor="white", linewidth=0.8, alpha=0.85)
ax1.axvline(lifespan_data.mean(), color="#e8734a", linestyle="--", linewidth=2.5,
            label=f"Laureate avg: {lifespan_data.mean():.1f} yrs")
ax1.axvline(us_life_exp, color="#2d6a9f", linestyle=":", linewidth=2.5,
            label=f"US avg (historical): {us_life_exp} yrs")
ax1.set_xlabel("Lifespan (years)", fontsize=11)
ax1.set_ylabel("Number of Laureates", fontsize=11)
ax1.set_title("Lifespan of Nobel Laureates
vs. General Population", fontsize=13, fontweight='bold')
ax1.legend()
ax1.spines[['top','right']].set_visible(False)

# Lifespan by category
lifespan_by_cat = nobel.groupby("category")["lifespan"].mean().dropna().sort_values(ascending=True)
ax2.barh(lifespan_by_cat.index, lifespan_by_cat.values, color="#4caf7d", edgecolor="white", alpha=0.85)
ax2.axvline(lifespan_data.mean(), color="#e8734a", linestyle="--", linewidth=2, label="Overall avg")
for i, (cat, val) in enumerate(lifespan_by_cat.items()):
    ax2.text(val + 0.2, i, f"{val:.1f}", va='center', fontsize=9, fontweight='bold')
ax2.set_xlabel("Mean Lifespan (years)", fontsize=11)
ax2.set_title("Average Lifespan by Nobel Category", fontsize=13, fontweight='bold')
ax2.set_xlim(lifespan_by_cat.min() - 5, lifespan_by_cat.max() + 5)
ax2.legend()
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig("q6_lifespan.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Laureate lifespan statistics (deceased only):")
print(f"  Mean:    {lifespan_data.mean():.1f} years")
print(f"  Median:  {lifespan_data.median():.1f} years")
print(f"  Longest: {lifespan_data.max():.1f} years")
print(f"  Shortest: {lifespan_data.min():.1f} years")
print(f"  Advantage over ~72yr historical avg: +{lifespan_data.mean()-72:.1f} years")


**Insight:** Nobel laureates live significantly longer than the historical average — often into their mid-80s. This is likely due to the "healthy worker effect": laureates tend to be highly educated, financially secure, and intellectually engaged — all factors associated with longevity.

---

### Question 7 — How has the geographic diversity of Nobel Prizes shifted over time?

The 20th century saw dramatic shifts in global power. Does the Nobel Prize reflect these geopolitical changes?


In [ ]:
# ── Q7: Geographic diversity over time ──
# Group by decade and count unique birth countries
geo_diversity = nobel.groupby("decade")["bornCountry"].nunique()

# Also track share of prizes won by USA vs rest of world
usa_share = nobel[nobel["bornCountry"].str.contains("United States", na=False)].groupby("decade").size()
total_per_decade = nobel.groupby("decade").size()
usa_pct = (usa_share / total_per_decade * 100).fillna(0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Unique countries per decade
ax1.bar(geo_diversity.index, geo_diversity.values, width=8, color="#2d6a9f", edgecolor="white", linewidth=1.5)
ax1.set_ylabel("# of Unique Birth Countries", fontsize=11)
ax1.set_title("Geographic Diversity of Nobel Laureates by Decade", fontsize=13, fontweight='bold')
ax1.set_xticks(geo_diversity.index)
ax1.set_xticklabels([f"{int(d)}s" for d in geo_diversity.index], rotation=45)
ax1.spines[['top','right']].set_visible(False)

# USA share over time
ax2.plot(usa_pct.index, usa_pct.values, "o-", color="#e8734a", linewidth=2.5, markersize=8)
ax2.fill_between(usa_pct.index, usa_pct.values, alpha=0.15, color="#e8734a")
ax2.set_xlabel("Decade", fontsize=11)
ax2.set_ylabel("% of Prizes to US-Born Laureates", fontsize=11)
ax2.set_title("Share of Nobel Prizes Won by US-Born Laureates", fontsize=13, fontweight='bold')
ax2.set_xticks(usa_pct.index)
ax2.set_xticklabels([f"{int(d)}s" for d in usa_pct.index], rotation=45)
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig("q7_geography.png", dpi=150, bbox_inches="tight")
plt.show()

print("Unique birth countries per decade:")
for decade, count in geo_diversity.items():
    print(f"  {int(decade)}s: {count} countries")


**Insight:** Geographic diversity has grown dramatically since the 1950s. Early Nobel Prizes were almost exclusively European; the post-WWII era saw rapid US dominance, followed by growing representation from Asia (particularly Japan and China) in recent decades. The Nobel Prize has become genuinely global.

---

### Question 8 — Which birth months are most common among Nobel laureates?

This is a quirky but fascinating question — sometimes called the "relative age effect" in sports. Does the same birthday bias exist in science?


In [ ]:
# ── Q8: Birth month analysis ──
nobel["birth_month"] = nobel["born_dt"].dt.month
month_counts = nobel["birth_month"].value_counts().sort_index()
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(11, 5))

colors_m = ["#c9a227" if v == month_counts.max() else "#2d6a9f" for v in month_counts.values]
bars = ax.bar(month_names, [month_counts.get(i, 0) for i in range(1,13)], 
              color=colors_m, edgecolor="white", linewidth=1.5)

# Expected if uniform
expected = len(nobel["birth_month"].dropna()) / 12
ax.axhline(expected, color="#e8734a", linestyle="--", linewidth=2, 
           label=f"Expected (uniform): {expected:.0f}")

for bar, val in zip(bars, [month_counts.get(i, 0) for i in range(1,13)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(val)), ha='center', fontsize=9, fontweight='bold')

ax.set_xlabel("Birth Month", fontsize=11)
ax.set_ylabel("Number of Laureates", fontsize=11)
ax.set_title("Nobel Laureates by Birth Month", fontsize=13, fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig("q8_birthmonth.png", dpi=150, bbox_inches="tight")
plt.show()

peak_month = month_counts.idxmax()
low_month  = month_counts.idxmin()
print(f"Peak birth month:   {month_names[peak_month-1]} ({month_counts[peak_month]} laureates)")
print(f"Lowest birth month: {month_names[low_month-1]} ({month_counts[low_month]} laureates)")
print(f"Expected per month: {expected:.0f}")


---
## The Story of Excellence: Insights & Conclusions

Our analysis of 120+ years of Nobel Prize data tells a rich, nuanced story about the nature of human achievement.

**Geography shapes genius — but immigration blurs borders.** The United States leads all countries in Nobel Prize count, but this masks an important reality: many American laureates were born in Germany, Eastern Europe, and Asia. The US didn't produce this genius so much as *attract* it through open universities, research funding, and political freedom. Countries that invest in welcoming foreign talent reap Nobel-level returns.

**Gender equity remains a work in progress.** Women represent barely 6% of all Nobel laureates — a sobering statistic for the 21st century. Progress is real but slow. The pace of change in the 2010s–2020s is the fastest in the prize's history, yet at this rate, parity remains decades away. The Physics category is the most striking outlier: in its 120-year history, fewer than 5 women have won.

**Science rewards patience.** The average laureate is nearly 60 years old when they win. This age has *increased* over time — a reflection of how modern science requires longer careers, bigger teams, and more cumulative work to produce breakthroughs worthy of the world's highest honor. The romanticized image of a young genius making a sudden discovery is not what the data shows.

**Extraordinary lives are long lives.** Nobel laureates live roughly 10–15 years longer than historical averages. Whether this is because intellectually rich lives promote longevity, or because wealth and status enable better healthcare, the pattern is clear: winning the Nobel Prize is correlated with living well into one's 80s.

**The world is catching up.** Early Nobel Prizes went almost exclusively to Europeans. By the 1980s, the US dominated. Today, the prize is increasingly global — with laureates from Japan, China, India, Africa, and Latin America. The Nobel Prize has become a genuine measure of global human achievement, not just Western academic prestige.

**The rarest achievement of all** belongs to Marie Curie: two Nobel Prizes, in two different sciences, in an era when women were barely admitted to universities. A century later, she remains the only person to achieve this. Her story isn't just about data — it's about what's possible when talent is finally given the chance to prove itself.

---

*Notebook created for Penn State DS 220 – Project #2 | Author: Jayden Beighey | https://github.com/jbeighey?tab=repositories *
